# Kaggle × Colab runner

Set `KAGGLE_API_TOKEN` (and `GITHUB_TOKEN` for a private repo) in 🔑 Secrets, edit the options, then run all.

In [ ]:
# @title Options
COMPETITION = "titanic"  # @param {type:"string"}
EXP = "exp001"  # @param {type:"string"}
ARGS = ""  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
USE_DRIVE = True  # @param {type:"boolean"}
# folder that holds this notebook on Drive; submissions go to <DRIVE_DIR>/<COMPETITION>/
DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"  # @param {type:"string"}
DOWNLOAD = True  # @param {type:"boolean"}
SUBMIT = False  # @param {type:"boolean"}

In [ ]:
# @title Setup
import os

from google.colab import userdata

REPO = "Moriyuki-S/kaggle_ws"
REPO_DIR = "/content/kaggle_ws"
DATA_DIR = f"/content/data/{COMPETITION}"

for k in ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY", "GITHUB_TOKEN"]:
    try:
        os.environ[k] = userdata.get(k)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        pass

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

token = os.environ.get("GITHUB_TOKEN")
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
if not os.path.exists(REPO_DIR):
    !git clone -q {url} {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH} && git log -1 --oneline

!pip install -q -U kaggle && pip install -q --no-deps -e .

if DOWNLOAD and not os.path.exists(DATA_DIR):
    !bash scripts/download.sh {COMPETITION} {DATA_DIR}

In [ ]:
# @title Run
import shutil
from datetime import datetime
from zoneinfo import ZoneInfo

from kaggle_ws import get_paths

sub = get_paths(COMPETITION).output / EXP / "submission.csv"
sub.unlink(missing_ok=True)  # never reuse a stale submission from a previous run

!python competitions/{COMPETITION}/exp/{EXP}/train.py {ARGS}
exit_code = _exit_code  # noqa: F821  (set by IPython after a ! command)
if exit_code != 0:
    raise RuntimeError(f"train.py failed (exit code {exit_code})")
if not sub.exists():
    raise FileNotFoundError(f"train.py did not write {sub}")

if USE_DRIVE:
    ts = datetime.now(ZoneInfo("Asia/Tokyo")).strftime("%Y%m%d_%H%M%S")
    dst = f"{DRIVE_DIR}/{COMPETITION}/submission_{ts}.csv"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    sub = shutil.copy(sub, dst)
    print("saved:", sub)

if SUBMIT:
    !kaggle competitions submit -c {COMPETITION} -f "{sub}" -m "{EXP} {ARGS}"